# Mount to Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Define the path
path = "/content/drive/MyDrive/Tutorial Session"

# Change the current working directory
os.chdir(path)

# Verify you are in the right spot
print("Current Directory:", os.getcwd())
print("Files in folder:", os.listdir())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current Directory: /content/drive/.shortcut-targets-by-id/15YxA3hPvK35rfyNAcfskPdPdJVhhRSjh/Tutorial Session
Files in folder: ['Section 1: Intro to LLM.ipynb', 'token_data.json', 'data', 'wandb', '__pycache__', 'KMGPT_API.py', 'AutoCriteria Demo.ipynb', 'Old Slides', 'qwen-example-lora', 'Biop Article.gdoc', 'Multi-Agent Demo Project.ipynb', 'ENAR Session 1.gslides', 'ENAR Session 2.gslides', 'Section 2: Biomedical Domain Adaptation.ipynb', 'Section 3: Autonomous Agents.ipynb', 'ENAR Session 3.gslides']


# Load Credentials

In [4]:
import json
with open('token_data.json','r') as f:
    token_dict = json.load(f)

HF_TOKEN = token_dict['HF_TOKEN']
OPENAI_TOKEN = token_dict['OPENAI_TOKEN']

# Replace the following with your own credentials
# HF_TOKEN =
# OPENAI_TOKEN =

# Example 1: Calling Directly from API

In [5]:
from openai import OpenAI

# Initialize the client
client = OpenAI(api_key=OPENAI_TOKEN)

## Comparison between system prompt and query

In [6]:
system_prompt = ""

query = '''
You are a helpful assistant for medical literature information extraction.
Your only task is to classify the treatment as Chemotherapy or Non-Chemotherapy.

ABSTRACT TEXT:
"The programmed cell death protein 1 (PD1) is one of the checkpoints that regulates the immune response.
Ligation of PD1 with its ligands PDL1 and PDL2 results in transduction of negative signals to T-cells.
PD1 expression is an important mechanism contributing to the exhausted effector T-cell phenotype.
The expression of PD1 on effector T-cells and PDL1 on neoplastic cells enables tumor cells to evade anti-tumor immunity.
Blockade of PD1 is an important immunotherapeutic strategy for cancers.
Pembrolizumab (Keytruda) is a humanized monoclonal anti-PD1 antibody that has been extensively investigated in numerous malignancies.
In melanoma refractory to targeted therapy, pembrolizumab induced overall response rates (ORRs) of 21-34%.
It was superior to another immune checkpoint inhibitor ipilimumab (Yervoy) in stage III/IV unresectable melanoma.
In refractory non-small cell lung cancer (NSCLC), pembrolizumab induced ORRs of 19-25%.
Based on these results, pembrolizumab was approved by the USA FDA for the treatment of advanced melanoma and NSCLC.
Tumor cell PDL1 expression may be a valid response predictor.
Molecular analysis also showed that tumors with high gene mutation burdens, which might result in the formation of more
tumor-related neo-antigens, had better responses to pembrolizumab. In malignancies including lymphomas and other solid tumors,
preliminary data showed that ORRs of around 20-50 % could be achieved. Adverse events occurred in up to 60% of patients,
but grade 3/4 toxicities were observed in <10% of cases. Immune-related adverse events including thyroid dysfunction,
hepatitis and pneumonitis are more serious and may lead to cessation of treatment.
"

Only return the name of this agent please.
Be sure to format it into a json as "treatment: xxxx".
Explain your rationale.
'''

response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ],
    temperature = 1
)

print(response.choices[0].message.content)

{"treatment": "Pembrolizumab"}

Rationale:
The abstract discusses the use of Pembrolizumab, a humanized monoclonal anti-PD1 antibody, as a crucial treatment for advanced melanoma and NSCLC, among other malignancies. This is a type of immunotherapy, which is considered a non-chemotherapy treatment. Therefore, the treatment mentioned in this context is Pembrolizumab, which falls under the category of Non-Chemotherapy treatment.


In [7]:
system_prompt ='''
You are a helpful assistant for medical literature information extraction.
Your only task is to classify the treatment as Chemotherapy or Non-Chemotherapy.
Do not return anything else.

Return the output as the following format:
{
    "treatment": "Non-Chemotherapy"
}
'''

query = '''
ABSTRACT TEXT:
"The programmed cell death protein 1 (PD1) is one of the checkpoints that regulates the immune response.
Ligation of PD1 with its ligands PDL1 and PDL2 results in transduction of negative signals to T-cells.
PD1 expression is an important mechanism contributing to the exhausted effector T-cell phenotype.
The expression of PD1 on effector T-cells and PDL1 on neoplastic cells enables tumor cells to evade anti-tumor immunity.
Blockade of PD1 is an important immunotherapeutic strategy for cancers.
Pembrolizumab (Keytruda) is a humanized monoclonal anti-PD1 antibody that has been extensively investigated in numerous malignancies.
In melanoma refractory to targeted therapy, pembrolizumab induced overall response rates (ORRs) of 21-34%.
It was superior to another immune checkpoint inhibitor ipilimumab (Yervoy) in stage III/IV unresectable melanoma.
In refractory non-small cell lung cancer (NSCLC), pembrolizumab induced ORRs of 19-25%.
Based on these results, pembrolizumab was approved by the USA FDA for the treatment of advanced melanoma and NSCLC.
Tumor cell PDL1 expression may be a valid response predictor.
Molecular analysis also showed that tumors with high gene mutation burdens, which might result in the formation of more
tumor-related neo-antigens, had better responses to pembrolizumab. In malignancies including lymphomas and other solid tumors,
preliminary data showed that ORRs of around 20-50 % could be achieved. Adverse events occurred in up to 60% of patients,
but grade 3/4 toxicities were observed in <10% of cases. Immune-related adverse events including thyroid dysfunction,
hepatitis and pneumonitis are more serious and may lead to cessation of treatment.
"

Only return the name of this agent please. Be sure to format it into a json as "treatment: xxxx". Explain your rationale.
'''

response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ],
    temperature = 1
)

print(response.choices[0].message.content)

{
    "treatment": "Non-Chemotherapy"
}


## Example 2: Temperature

In [8]:
system_prompt = "You are helpful assistant for literature summerization."

query = '''
Please summary the following abstract in no more than 3 sentences.

BSTRACT TEXT:
"The programmed cell death protein 1 (PD1) is one of the checkpoints that regulates the immune response.
Ligation of PD1 with its ligands PDL1 and PDL2 results in transduction of negative signals to T-cells.
PD1 expression is an important mechanism contributing to the exhausted effector T-cell phenotype.
The expression of PD1 on effector T-cells and PDL1 on neoplastic cells enables tumor cells to evade anti-tumor immunity.
Blockade of PD1 is an important immunotherapeutic strategy for cancers.
Pembrolizumab (Keytruda) is a humanized monoclonal anti-PD1 antibody that has been extensively investigated in numerous malignancies.
In melanoma refractory to targeted therapy, pembrolizumab induced overall response rates (ORRs) of 21-34%.
It was superior to another immune checkpoint inhibitor ipilimumab (Yervoy) in stage III/IV unresectable melanoma.
In refractory non-small cell lung cancer (NSCLC), pembrolizumab induced ORRs of 19-25%.
Based on these results, pembrolizumab was approved by the USA FDA for the treatment of advanced melanoma and NSCLC.
Tumor cell PDL1 expression may be a valid response predictor.
Molecular analysis also showed that tumors with high gene mutation burdens, which might result in the formation of more
tumor-related neo-antigens, had better responses to pembrolizumab. In malignancies including lymphomas and other solid tumors,
preliminary data showed that ORRs of around 20-50 % could be achieved. Adverse events occurred in up to 60% of patients,
but grade 3/4 toxicities were observed in <10% of cases. Immune-related adverse events including thyroid dysfunction,
hepatitis and pneumonitis are more serious and may lead to cessation of treatment.
"

'''


### Temperature at 1

In [9]:
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ],
    temperature = 1
)

print(response1.choices[0].message.content)

The programmed cell death protein 1 (PD1) is a key immune checkpoint that, when engaged by its ligands PDL1 and PDL2, inhibits T-cell activity and allows tumor cells to evade immune detection. Pembrolizumab, a monoclonal anti-PD1 antibody, has shown significant efficacy in treating advanced melanoma and non-small cell lung cancer (NSCLC), with overall response rates of 21-34% and 19-25%, respectively, leading to its FDA approval for these malignancies. While pembrolizumab can achieve substantial response rates in various cancers, it is associated with adverse events in up to 60% of patients, primarily immune-related issues that may necessitate discontinuation of therapy.


In [10]:
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ],
    temperature = 1
)

print(response2.choices[0].message.content)

The programmed cell death protein 1 (PD1) plays a crucial role in regulating the immune response by promoting T-cell exhaustion and allowing tumor cells to evade anti-tumor immunity. Pembrolizumab, an anti-PD1 monoclonal antibody, has demonstrated significant effectiveness in treating various malignancies, particularly advanced melanoma and non-small cell lung cancer, resulting in overall response rates of 21-34% and 19-25%, respectively. While pembrolizumab shows promise, it can lead to adverse events in a substantial number of patients, highlighting the need for careful monitoring during treatment.


### Temperature at 0

In [11]:
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ],
    temperature = 0
)

print(response1.choices[0].message.content)

The programmed cell death protein 1 (PD1) plays a crucial role in regulating immune responses and contributes to T-cell exhaustion, allowing tumor cells to evade anti-tumor immunity. Pembrolizumab, a monoclonal anti-PD1 antibody, has shown significant efficacy in treating advanced melanoma and non-small cell lung cancer, with overall response rates ranging from 19-34%, leading to its FDA approval. While the treatment can lead to immune-related adverse events in up to 60% of patients, serious toxicities are relatively rare, occurring in less than 10% of cases.


In [12]:
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ],
    temperature = 0
)

print(response2.choices[0].message.content)

The programmed cell death protein 1 (PD1) plays a crucial role in regulating immune responses and is associated with T-cell exhaustion, allowing tumor cells to evade anti-tumor immunity. Pembrolizumab, a monoclonal anti-PD1 antibody, has shown significant efficacy in treating advanced melanoma and non-small cell lung cancer, with overall response rates ranging from 19-34%, leading to its FDA approval. While the treatment can lead to adverse events in up to 60% of patients, serious toxicities are relatively rare, and tumor PDL1 expression and high gene mutation burdens may serve as predictors for better responses.


## LangChain

### Installation

In [13]:
%pip install -U langchain-openai langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 23.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1


# Local Deployment

## Create Pipeline

In [14]:
from transformers import pipeline
import torch

# Initialize the pipeline
# We use "text-generation" for Qwen models
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen3-0.6B",
    torch_dtype="auto",
    device_map="auto",
    token = HF_TOKEN
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [15]:
system_prompt = "You are helpful assistant for literature summerization."

query = '''
Please summarize the following abstract in no more than 3 sentences.

BSTRACT TEXT:
"The programmed cell death protein 1 (PD1) is one of the checkpoints that regulates the immune response.
Ligation of PD1 with its ligands PDL1 and PDL2 results in transduction of negative signals to T-cells.
PD1 expression is an important mechanism contributing to the exhausted effector T-cell phenotype.
The expression of PD1 on effector T-cells and PDL1 on neoplastic cells enables tumor cells to evade anti-tumor immunity.
Blockade of PD1 is an important immunotherapeutic strategy for cancers.
Pembrolizumab (Keytruda) is a humanized monoclonal anti-PD1 antibody that has been extensively investigated in numerous malignancies.
In melanoma refractory to targeted therapy, pembrolizumab induced overall response rates (ORRs) of 21-34%.
It was superior to another immune checkpoint inhibitor ipilimumab (Yervoy) in stage III/IV unresectable melanoma.
In refractory non-small cell lung cancer (NSCLC), pembrolizumab induced ORRs of 19-25%.
Based on these results, pembrolizumab was approved by the USA FDA for the treatment of advanced melanoma and NSCLC.
Tumor cell PDL1 expression may be a valid response predictor.
Molecular analysis also showed that tumors with high gene mutation burdens, which might result in the formation of more
tumor-related neo-antigens, had better responses to pembrolizumab. In malignancies including lymphomas and other solid tumors,
preliminary data showed that ORRs of around 20-50 % could be achieved. Adverse events occurred in up to 60% of patients,
but grade 3/4 toxicities were observed in <10% of cases. Immune-related adverse events including thyroid dysfunction,
hepatitis and pneumonitis are more serious and may lead to cessation of treatment.
"

'''

## Create Response

In [16]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": query}
]

# Generate
# The pipeline handles tokenization and decoding internally
response = pipe(messages, max_new_tokens=512, temperature = 0.1)

# Extract and print the content
print(response[0]['generated_text'][-1]['content'])

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>
Okay, let's see. The user wants a summary of the abstract in three sentences. The original text is about PD1, its role in immune response, how it's involved in cancer, and the effectiveness of pembrolizumab.

First, I need to capture the main points: PD1 as a checkpoint, its ligands, and the role in immune evasion. Then mention pembrolizumab's success in melanoma and NSCLC. Also, note the adverse effects. Let me check if I can fit all that into three concise sentences without being too wordy. Make sure each sentence flows and covers all key points.
</think>

The programmed cell death protein 1 (PD1) acts as an immune checkpoint, linking tumor cells to T-cells by transducing negative signals. Its expression on effector T-cells and PDL1 on neoplastic cells enables tumor evasion. Pembrolizumab (Keytruda) improves immune response in refractory melanoma and NSCLC, with 21–34% ORRs and 19–25% in NSCLC, offering a better therapeutic option than ipilimumab. Adverse effects were common,